In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


In [ ]:
df = pd.read_csv("../data/raw/spotify-tracks-dataset-detailed.csv")
pd.set_option("display.max_columns", None)

In [ ]:
# dataset audit
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# missing value audit
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
})
# sort missing values
missing = missing.sort_values("missing_pct", ascending=False)
missing

In [ ]:
# duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"Duplicated rows: {duplicate_rows:,}")
# duplicate track IDs
duplicate_track_ids = df["track_id"].duplicated().sum()
print(f"Duplicate track IDs: {duplicate_track_ids:,}")

In [ ]:
# investigate duplicate IDs
duplicate_tracks = df[
    df["track_id"].duplicated(keep=False)
].sort_values("track_id")
duplicate_tracks.head(20)

In [ ]:
# check whether duplicated IDs contain conflicted values
if duplicate_track_ids > 0:
    consistency = duplicated_tracks.groupby("track_id").nunique()
    consistency
else:
    print("No duplicate track IDs found")

In [ ]:
# candidate features
audio_features = [
    "danceability",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "valence",
    "tempo",
    "loudness",
    "duration_ms",
    "key",
    "mode",
    "time_signature"
]

In [ ]:
# check feature dtypes
df[audio_features].dtypes

In [ ]:
# check non-numeric features
non_numeric = df[audio_features].select_dtypes(
    exclude=np.number
).columns
print("Non-numeric columns:", list(non_numeric))

In [ ]:
# check bounded features
bounded_features = [
    "danceability",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "valence"
]

for feature in bounded_features:
    invalid = ((df[feature] < 0) | (df[feature] > 1)).sum()
    print(f"{feature}: {invalid:,} invalid values")

In [ ]:
# check physical constraints
print("Negative durations:", (df["duration_ms"] < 0).sum())
print("Non-positive tempos:", (df["tempo"] <= 0).sum())

In [ ]:
# audit summary
audit = {
    "rows": len(df),
    "columns": df.shape[1],
    "duplicate_rows": df.duplicated().sum(),
    "duplicate_track_ids": df["track_id"].duplicated().sum(),
    "total_missing_values": df.isna().sum().sum(),
    "non_numeric_audio_features": len(non_numeric)
}

pd.Series(audit)